# 现代 LLM 推理系统

> 前面四章都在优化「一个请求」：怎么选 Token、为什么慢、怎么压小、怎么猜得更快。线上服务面对的却是另一种局面——100 个请求同时到达：有的 5 个 Token 就说完，有的要写 2000 字；有的共享同一段 system prompt，有的带着 100K 的长文档。让它们挤进同一张 GPU，谁先算？
>
> 这一章就是 vLLM、SGLang、TensorRT-LLM 这些推理引擎真正在做的事。这些名词听起来像是一门黑话，但学完这一节你会发现：`Continuous Batching`、`PagedAttention`、`Prefix Caching`、`Chunked Prefill`、`PD 分离`，每一个背后都是一个非常具体、你能说清楚的工程问题。
>
> 本章六块内容：
>
> 1. **指标**：吞吐、TTFT、TPOT——谈优化先对准指标。
> 2. **调度**：Static Batching 的浪费与 Continuous Batching。
> 3. **显存**：PagedAttention，KV Cache 的分页管理。
> 4. **复用**：Prefix Caching 与 RadixAttention。
> 5. **公平**：Chunked Prefill，不让长请求堵车。
> 6. **分池**：PD 分离，以及 Kernel 层、多卡并行的名词归位。

## 1. 吞吐、TTFT 与 TPOT

单请求时代只有「快慢」一个维度。服务化之后，至少要三个指标才能描述一次服务的全貌。

**Throughput（吞吐）**是系统每秒产出的总 Token 数——所有请求加在一起。它决定成本：同样一张卡，吞吐翻倍等于单 Token 成本减半。**TTFT** 是用户发问后到第一个字出现的等待——用户对它的忍耐以秒计。**TPOT** 是之后每个 Token 的间隔——决定「出字节奏」是否流畅。

三个指标互相牵制。把并发拉高，多个请求共享一次权重搬运，吞吐上升；但每个请求分到的计算变碎，TTFT 和 TPOT 通常变差。所以「vLLM 比 SGLang 快 30%」这种结论没有意义——快的是吞吐还是延迟？P50 还是 P99？后面评测一本会专门讲怎么公平比较，这里先把指标立住：

```text
Prefill  -> 主要影响 TTFT
Decode   -> 主要影响 TPOT
两者一起 -> 决定吞吐
```

另外记住 P50 / P95 / P99：平均值会骗人，1 个被饿 10 秒的请求不会出现在平均延迟里，但会出现在用户的差评里。

## 2. 从 Static Batching 到 Continuous Batching

先回答一个更基本的问题：为什么要把多个请求拼成一个 batch？

上一本的结论是 Decode 每步都要把整套权重从显存搬进计算单元，却只服务 1 个 Token——带宽利用率极低。把 32 个请求拼起来，权重还是搬一遍，却同时服务 32 个 Token。batching 的本质就在这里：**让一次搬运服务更多人**。先看数字量级：

In [ ]:
params = 7e9
gpu_tflops = 312
gpu_bw_gbs = 2000

flops_per_token = 2 * params
compute_tokens_s = gpu_tflops*1e12/flops_per_token

weight_bytes = params*2
bandwidth_single_stream = gpu_bw_gbs*1e9/weight_bytes

print("compute roofline proxy:", round(compute_tokens_s), "token/s")
print("single-stream weight-bandwidth proxy:", round(bandwidth_single_stream), "token/s")
print("注意：这只是解释为何 decode 容易 memory-bound 的数量级直觉。")


两个极限差了 7 倍：纯算力上限约 22000 Token/s，而单请求把权重搬完只能支撑约 140 Token/s。Decode 的现实就贴着下面那条线——把 batch 拉大，才能向上面的线靠。

但怎么拼 batch，有两个流派。**Static Batching**（静态批）最直觉：凑够一批，整批一起进、一起出。它的问题在「一起出」——长短不齐的请求里，短的要陪长的空转到最长的算完；批没结束，新请求也进不来。

**Continuous Batching**（连续批）换了一种思路：调度粒度从「整批」细化到「每一步」（iteration-level scheduling）——任何一个 Decode step 结束时，完成的请求立刻退出、空出的位置立刻交给等待中的新请求。

用一个小模拟器把两种调度各跑一遍。6 个请求，生成长度 5 到 200 不等，同时最多容纳 4 个：

In [ ]:
names = ["A", "B", "C", "D", "E", "F"]
lengths = [5, 200, 8, 150, 6, 180]   # 每个请求要生成的 token 数
batch_size = 4

def simulate_static(lengths, batch_size):
    """静态批：凑一批一起跑，整批被最长的请求拖到一起结束"""
    schedule = []
    t = 0
    for i in range(0, len(lengths), batch_size):
        group = lengths[i:i + batch_size]
        for j in range(len(group)):
            schedule.append((i + j, t, t + max(group)))
        t += max(group)
    return schedule

def simulate_continuous(lengths, batch_size):
    """连续批：任何一步只要有空位，等待中的请求立刻补上"""
    schedule = [None] * len(lengths)
    pending = list(range(len(lengths)))
    remaining = {i: l for i, l in enumerate(lengths)}
    active = {}
    t = 0
    while pending or active:
        while pending and len(active) < batch_size:
            rid = pending.pop(0)
            active[rid] = t
        for rid in list(active):
            remaining[rid] -= 1
            if remaining[rid] == 0:
                schedule[rid] = (rid, active[rid], t + 1)
                del active[rid]
        t += 1
    return schedule

static_sched = simulate_static(lengths, batch_size)
cont_sched = simulate_continuous(lengths, batch_size)

print("请求生成长度:", dict(zip(names, lengths)), " batch_size =", batch_size)
print()
for name, (rid, s, e) in zip(names, static_sched):
    print(f"静态批  {name}: step {s:>3} - {e:>3}")
print("静态批  总步数:", max(e for _, s, e in static_sched))
print()
for name, (rid, s, e) in zip(names, cont_sched):
    print(f"连续批  {name}: step {s:>3} - {e:>3}")
print("连续批  总步数:", max(e for _, s, e in cont_sched))
print()
print("关键观察：静态批里 A/C 和 200-token 的 B 锁死一批，5 个 token 的 A 也要等 200 步；")
print("连续批里 A 提前完成时 E 立刻补位，F 也不用等第二批，总时间从 380 步降到 200 步")


In [ ]:
# 甘特图：上=静态批（整批对齐），下=连续批（完成即补位）
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(7, 4.5), sharex=True)
panels = [(axes[0], static_sched, "Static batching (batch locked until longest finishes)"),
          (axes[1], cont_sched, "Continuous batching (slot refilled immediately)")]
for ax, sched, title in panels:
    for rid, s, e in sched:
        ax.barh(rid, e - s, left=s, height=0.6, color="tab:blue", alpha=0.8)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=10)
axes[1].set_xlabel("decode step")
plt.tight_layout()
plt.show()

对着甘特图看两处细节。上面一张：A 只要 5 步，却陪着 B 空转到 200 步；E 明明第 5 步就能开始，被「批」挡到 205 步。下面一张：同样的 6 个请求，A 提前完成时 E 立刻补位，F 不用等第二批——总时间从 380 步降到 200 步。

「批」从静态的容器，变成了动态的槽位。这就是 Continuous Batching 在调度的事，也是 vLLM 论文里的第一个卖点。

但调度解决的是「谁来算」；这些请求算出来的 KV Cache 放在哪，是下一个问题——而且是个显存问题。

## 3. PagedAttention 与 KV Cache 分页

每个活跃请求都有一份 KV Cache，而且长度各不相同。传统的分配方式是给每个请求预留一段**连续**显存，按最大上下文长度申请。两个浪费立刻出现：一是预留浪费——按 512 Token 申请、实际只用 50，多出来的 462 个位置白占；二是碎片——请求不断进出，显存里留下大小不一的空洞，新的整段申请放不进去。

操作系统在内存管理上遇到过一模一样的问题，解法也是现成的：**分页**。PagedAttention 把 KV Cache 切成固定大小的 block（比如 16 Token 一页），按需分配、页表映射——序列在逻辑上连续，在物理显存里离散存放。

先量化一下「预留 vs 分页」到底差多少：

In [ ]:
# 模拟两种 KV Cache 分配：按 max_len 整段预留 vs 按 16-token 分页按需分配
seqs = [("req1", 50), ("req2", 300), ("req3", 120), ("req4", 45)]
max_len = 512            # 整段预留时要按最大上下文申请
page_size = 16           # PagedAttention 风格的页大小
kb_per_token = 2         # 每个 token 的 KV Cache 字节数（示意值）

actual = sum(l * kb_per_token for _, l in seqs)
reserved = len(seqs) * max_len * kb_per_token
paged = sum(-(-l // page_size) * page_size * kb_per_token for _, l in seqs)

print(f"实际需要的 KV:      {actual:>5} KB")
print(f"整段预留(max_len): {reserved:>5} KB，浪费 {reserved - actual} KB")
print(f"按页分配(16/页):   {paged:>5} KB，浪费 {paged - actual} KB")
print()
print("关键观察：分页把浪费限制在「每个请求最多不到一页」；")
print("页越小浪费越少，但页表和映射开销上升——这是操作系统分页的老权衡")

In [ ]:
# 整段预留 vs 分页 vs 实际需要的显存对比
import matplotlib.pyplot as plt

labels = ["reserved (max_len)", "actual KV needed", "paged (16 tokens/page)"]
values = [reserved / 1024, actual / 1024, paged / 1024]

plt.figure(figsize=(6, 3.2))
plt.bar(labels, values, color=["tab:red", "tab:green", "tab:blue"])
plt.ylabel("KV cache memory (MB)")
plt.title("Paged allocation cuts the waste of reserving max_len")
for i, v in enumerate(values):
    plt.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
plt.show()

同样是 4 个请求，整段预留浪费了 3 GB 出头；分页把浪费压到每个请求最多不到一页（这里 29 KB 的量级）。省出来的显存意味着能容纳更多并发——调度一章的连续批才有槽位可用。

有一句澄清必须说：**PagedAttention 没有改 Attention 的计算公式**，它管的是 KV Cache 这块缓存的内存管理。它也不是 FlashAttention 的亲戚——后者优化的是 Attention 计算本身的读写（本章第 7 节）。名字里都有 Attention，层次完全不同。

分页解决「怎么放」，还有一个更省的问题：有些 KV 根本不用重算。

## 4. Prefix Caching 与 RadixAttention

线上流量有一个显著特征：大量请求共享同一段开头。同一个应用的几千个请求带着同一条 system prompt；同一份文档被反复追问；多轮对话每一轮都重发完整历史。每来一个请求，引擎都要对这段相同的前缀做一次完整的 Prefill——算的东西一模一样，结果也一模一样。

**Prefix Cache** 的做法：把算好的前缀 KV 留在显存里，后续请求命中相同前缀就直接跳过这段 Prefill。命中的部分越大，TTFT 越接近零。

一个新的问题随之而来：不同请求的前缀共享是「树状」的——`ABCDEF`、`ABCDXY`、`ABCZZ` 共享 `ABC`，前两个还额外共享 `ABCD`。怎么组织这些前缀，让任意新请求都能快速找到最长可复用前缀？SGLang 的 **RadixAttention** 用的就是 radix tree（基数树）——把它想成一棵「按 Token 分叉的字典树」，公共路径只存一份。算一笔账：

In [ ]:
# 前缀共享：三个请求的 token 序列放进 trie，公共前缀只需算一次 Prefill
seqs = [list("ABCDEF"), list("ABCDXY"), list("ABCZZ")]

def count_trie_tokens(seqs):
    """把所有序列放入 trie，返回树里存了多少 token（= 需要算 prefill 的 token 数）"""
    root, total = {}, 0
    for seq in seqs:
        node = root
        for tok in seq:
            if tok not in node:
                node[tok] = {}
                total += 1      # 第一次出现的 token 才需要算一次 prefill
            node = node[tok]
    return total

naive_total = sum(len(s) for s in seqs)
radix_total = count_trie_tokens(seqs)

print("三个请求:", ["".join(s) for s in seqs])
print(f"不做共享:      {naive_total} 个 token 各算各的 Prefill")
print(f"radix 共享后:  {radix_total} 个 token 的 Prefill（节省 {naive_total - radix_total}）")
print()
print("关键观察：system prompt 越长、并发请求越多，前缀复用省下的 Prefill 越可观")

In [ ]:
# 每个请求独立 Prefill vs 前缀树共享后的 Prefill 工作量
import matplotlib.pyplot as plt

plt.figure(figsize=(4.8, 3))
plt.bar(["no sharing", "radix tree"], [naive_total, radix_total],
        color=["tab:red", "tab:green"])
plt.ylabel("prefill tokens to compute")
plt.title("Shared prefixes are computed once")
for i, v in enumerate([naive_total, radix_total]):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.show()

18 个 Token 的 Prefill 工作量，radix 树把它压到 10 个——节省的 8 个全部来自共享前缀。system prompt 越长、并发越大，这笔账越可观；多轮对话场景里，历史部分的 Prefill 几乎全部命中，每轮只需为新增的几个 Token 付费。

到这里，单机单卡的「算、存、复用」都齐了。剩下的问题出在时间维度上。

## 5. Chunked Prefill

连续批让调度灵活了，但有一种请求能把它重新堵死：一个 100K Token 的长 Prompt。它的一次 Prefill 要独占 GPU 几百毫秒，期间所有在线 Decode 请求全部排队——用户正看着字往外蹦，突然卡住半秒。这类「队头阻塞」（head-of-line blocking）是 TPOT 长尾的主要来源。

**Chunked Prefill** 的解法很直接：把长 Prompt 切成若干块（chunk），一块一块地 Prefill，块与块之间穿插执行别的请求的 Decode step。长请求的总 Prefill 时间差不多，但别人的出字节奏保住了。

它调的是 TTFT 和 TPOT 的平衡：chunk 切小，Decode 不卡，但长请求自己的 TTFT 略增；chunk 切大则相反。这是调度器的一个真实旋钮，vLLM 里对应 `chunked_prefill_size` 这类配置。

## 6. Prefill / Decode 分离

现在退后一步看整条链路，Prefill 和 Decode 的脾气完全相反：

```text
Prefill: 大矩阵，吃算力   -> 决定 TTFT
Decode : 小步串行，吃带宽 -> 决定 TPOT
```

放在同一组 GPU 上，这两种负载会互相挤占：按 Prefill 配算力，Decode 时代算力闲置；按 Decode 配带宽，Prefill 又排长队。**PD 分离（Prefill-Decode Disaggregation）**的思路就是把它们拆到两组不同的 worker：Prefill 池专心吃算力，Decode 池专心吃带宽，请求在两池之间靠 **KV Transfer**（把 Prefill 算好的 KV Cache 传给 Decode 侧）接起来。

```text
Gateway
   ↓
Prefill workers  --KV Transfer-->  Decode workers
                                        ↓
                                 Streaming response
```

注意它不是「把 Transformer 拆成两半」——同一个请求的两个阶段，做的是**资源解耦**。真实的难点也全在接缝上：KV 传输本身要吃网络带宽；两池的配比（几张卡 Prefill、几张卡 Decode）随流量变化；请求路由、缓存归属、故障恢复都要重新设计。厂商说「支持 PD 分离」，说的就是这整套架构。

一个重要的判断：PD 分离首先是为了**独立调 TTFT / TPOT、控制长尾延迟**，不应简单理解成「必然提高吞吐」——KV 传输本身是有成本的。

## 7. FlashAttention、FlashInfer 与 CUDA Graph

前面六节说的都是内存管理和调度——系统层的机制。厂商报告里还有一组更靠近执行层的词，值得单独归位：

- **FlashAttention / FlashInfer / FlashMLA**：优化 Attention kernel 本身。核心思路是不把巨大的中间注意力矩阵写回显存，分块计算、原地归并——省的是 HBM 读写。FlashInfer 进一步把这些 kernel 打包成推理专用库
- **Fused Kernel**：把多个小算子融合成一个——每次 kernel 启动和中间结果落盘都是开销，融合后一并省掉
- **CUDA Graph**：Decode 循环每步的 kernel 序列几乎相同，逐个启动的开销（launch overhead）在 GPU 空闲时反而成为瓶颈。把整段执行图 capture 一次、之后整体 replay，启动开销只剩一次
- **torch.compile / 图优化**：从计算图层面做融合和特化，思路同上

所以「PagedAttention + FlashAttention」不是重复建设——一个管 KV Cache 怎么放，一个管 Attention 怎么算：

```text
PagedAttention -> KV Cache 的内存管理（系统层）
FlashAttention -> Attention 计算的读写效率（Kernel 层）
CUDA Graph     -> 重复 kernel 的启动开销（运行时层）
```

## 8. TP、PP、DP、EP 与 CP

单卡放不下模型、或算不动流量时，就要切多卡。五个缩写对应五个不同的切分维度：

| 名词 | 切的对象 | 一句话直觉 |
|:---|:---|:---|
| Tensor Parallel（TP） | 层内的矩阵 | 一层太大，多卡一起算 |
| Pipeline Parallel（PP） | Transformer 层 | 不同层放不同卡 |
| Data Parallel（DP） | 模型副本 | 多副本分担更多请求 |
| Expert Parallel（EP） | MoE 的专家 | 不同专家放不同卡 |
| Context Parallel（CP） | 序列维度 | 超长上下文跨卡处理 |

推理场景最常用 TP（`--tensor-parallel-size 2` 就是它）。判断一个并行方案，真正要问的还是那个老问题：**模型参数、KV Cache、Token 和通信分别落在哪些卡上？通信开销会不会吃掉并行收益？** 工程里这些策略可以组合，附录「并行策略」一本有展开。

## 9. 术语地图

现在把这一章的所有名词放进一张表。以后在报告或 JD 里看到任何一个，先回忆它对应哪一节的问题：

| 名词 | 它在解决什么 |
|:---|:---|
| Continuous Batching | 调度：短请求不陪跑，空位立刻补人 |
| PagedAttention | 显存：KV Cache 分页，消灭预留浪费和碎片 |
| Prefix Caching | 复用：相同前缀不重算 Prefill |
| RadixAttention | 复用：树状组织共享前缀 |
| Chunked Prefill | 公平：长 Prompt 不堵死在线 Decode |
| PD 分离 / KV Transfer | 资源：两种负载分池，KV 跨池传输 |
| FlashAttention / FlashInfer | Kernel：Attention 的读写效率 |
| CUDA Graph | 运行时：Decode 循环的启动开销 |
| TP / PP / DP / EP / CP | 多卡：五个切分维度 |
| Speculative Decoding | 串行：一次 forward 确认多个 Token |
| KV Cache 量化 | 显存：KV 本身的低比特 |

拿一条真实的招聘 JD 检验一下：

> 熟悉 vLLM / SGLang，理解 PagedAttention、Continuous Batching、Prefix Caching、Chunked Prefill、Speculative Decoding、PD Disaggregation。

现在这条 JD 里的每个词，你都能说出它在系统里的位置、解决的问题、影响的指标——这就是这一章的目标。

## 小结

- 服务化之后至少盯三个指标：吞吐（成本）、TTFT（等待）、TPOT（节奏）；长尾看 P95/P99
- Static Batching 让短请求陪跑；Continuous Batching 按 step 调度，空位即补
- PagedAttention 用分页管理 KV Cache——省的是预留浪费和碎片，没改 Attention 公式
- Prefix Caching / RadixAttention 让共享前缀只算一次 Prefill
- Chunked Prefill 切长 Prompt，防队头阻塞，调 TTFT 与 TPOT 的平衡
- PD 分离按负载特征分池，靠 KV Transfer 接起来——为长尾和配比，不为吞吐本身
- FlashAttention / CUDA Graph 在 Kernel 与运行时层；TP/PP/DP/EP/CP 是五个切分维度

系统变快之后，还剩一个问题没人回答——快出来的这些，质量掉没掉？这正是下一章：

> **推理系统变快了、模型被量化了，怎么证明输出质量没有变化、对比是公平的？**

## 作业

三道题对应三个模拟器的核心机制：调度补位、分页浪费、前缀匹配。

> **关于 AI 辅助**：可以让 AI 提示思路、拆解步骤，但不建议直接让 AI 完成题目。
> 这三个机制是读 serving 系统源码前的最小模型。

### 作业 1：写一个 Continuous Batching 的单步调度

`active` 是还在跑的请求 `[[id, 剩余token], ...]`，`pending` 是等待队列。一步调度 = 先补位、
再让所有活跃请求剩余量减一，减到 0 的请求在本步结束时退出。

**小提示**：先用 `if` 判断有没有空位，把 `pending[0]` 弹出加入 `active`，再统一减一。

In [ ]:
# 作业 1：连续批调度单步 填空

def one_step(active, pending, batch_size):
    """返回 (新的 active, 本步完成的请求 id 列表)"""
    if pending and len(active) < batch_size:
        # TODO：把下面三引号里的内容替换成你的代码
        """把 pending 的第一个请求弹出并加入 active"""
    finished = []
    for item in active:
        item[1] -= 1
        if item[1] == 0:
            finished.append(item[0])
    return [item for item in active if item[1] > 0], finished

active, done = one_step([], [[1, 5]], 4)
assert [i[0] for i in active] == [1] and [i[1] for i in active] == [4]
assert done == []
active, done = one_step([[1, 1]], [[2, 3]], 2)
assert done == [1]
assert [i[0] for i in active] == [2] and [i[1] for i in active] == [2]
print("✅ 作业 1 通过：你已经写出连续批调度的核心一步")

### 作业 2：算分页的浪费

按 `page_size` 分页时，长度为 `length` 的序列实际占用
`ceil(length / page_size) * page_size` 个 token 位——多 1 个 token 也要多占一整页。

**小提示**：向上取整可以写成 `-(-a // b)`。

In [ ]:
# 作业 2：分页浪费 填空

def paged_tokens(length, page_size=16):
    """返回按 page_size 分页后实际占用的 token 位"""
    # TODO：把下面三引号里的内容替换成你的代码
    """对 length / page_size 向上取整再乘 page_size"""

assert paged_tokens(16, 16) == 16      # 正好整页，零浪费
assert paged_tokens(17, 16) == 32      # 多 1 个 token 也要多占一整页
assert paged_tokens(45, 16) == 48
waste = sum(paged_tokens(l) - l for l in [50, 300, 120, 45])
assert waste == 29, waste
print("✅ 作业 2 通过：分页把浪费限制在「每个请求最多不到一页」")

### 作业 3：在 radix tree 里找最长可复用前缀

新请求到来时，先在前缀树里找能复用的最长前缀，只有匹配不到的部分才需要新的 Prefill。

**小提示**：从根出发逐 token 往下走，遇到当前节点没有的 token 就停。

In [ ]:
# 作业 3：最长前缀匹配 填空

def build_trie(seqs):
    root = {}
    for seq in seqs:
        node = root
        for tok in seq:
            node = node.setdefault(tok, {})
    return root

def longest_prefix_length(root, seq):
    """返回 seq 在 trie 中可复用的前缀长度"""
    length = 0
    node = root
    for tok in seq:
        # TODO：把下面三引号里的内容替换成你的代码
        """tok 在 node 里就往下走并累加，否则 break"""
    return length

trie = build_trie([list("ABCDEF"), list("ABCDXY"), list("ABCZZ")])
assert longest_prefix_length(trie, list("ABCDQ")) == 4
assert longest_prefix_length(trie, list("ABCZ")) == 4
assert longest_prefix_length(trie, list("XYZ")) == 0
print("✅ 作业 3 通过：新请求只需为匹配不到的部分付 Prefill")